notebook para testar a geracao de um df a apartir de uma matriz (imagem) e executar o pca em spark

# Imports

In [1]:
import os
import gc # to free memory
import numpy as np
from math import ceil
import pandas as pd
import psutil
import time
import sys
import glob

import pickle
import random
from tqdm import tqdm

import multiprocessing
nproc = multiprocessing.cpu_count()-2

In [2]:
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
import pyspark.pandas as ps

In [3]:
#imports for spark
from pyspark.sql import SparkSession

In [4]:
#imports for spark, cont.
from pyspark.sql import functions as F
from pyspark.sql.functions import when, col, array, udf
from pyspark.ml.feature import PCA
from pyspark.ml.feature import PCAModel

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.linalg import Vectors

In [5]:
# Step 1: Start a Spark ssession
spark = SparkSession.builder \
    .appName("Pandas to PySpark DataFrame") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/14 16:48:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Code

## Gen df from matrix

In [46]:

matrix = np.array([[[1.0, 2.0, 3.0,4.0], [-9999.0, 6.0, -7.0, 8.0], [9.0, 10.0, 11.0, -9999.0]],
                   [[13.0,14.0, 15.0, 16.0], [17, 18, 19, 20], [21.0, 22.0, 23.0, 24.0]],
                   [[25.0, 26.0, 27.0, 28.0], [29.0, 30.0, 31.0, 32.0], [33.0, -32768.0, 35.0, 36.0]]])
matrix2 = np.array([[[-9999.0, 22.0, 23.0,24.0], [25.0, 26.0, -27.0, 28.0], [29.0, 40.0, 41.0, -9999.0]],
                   [[43.0,44.0, 45.0, 46.0], [47, 48, 49, 50], [51.0, 52.0, 53.0, 54.0]],
                   [[55.0, 56.0, 57.0, 58.0], [59.0, 70.0, 71.0, 72.0], [73.0, -32768.0, 75.0, 76.0]]])

In [8]:
bands_name =['b1t1', 'b2t1', 'b3t1', 'b4t1']
bands_name2 =['b1t2', 'b2t2', 'b3t2', 'b4t2']

In [47]:
sh_print=1
num_rows, num_cols, cols_df = matrix.shape
print (f'matrix rows={num_rows}, cols={num_cols}, bands={cols_df}') if sh_print else None
# Flatten the matrix values and reshape into a 2D array
valuesM = matrix.reshape(-1, 4)
valuesM2 = matrix2.reshape(-1, cols_df)

matrix rows=3, cols=3, bands=4


In [48]:
sdftM = ps.DataFrame(valuesM, columns=bands_name)
sdftM2 = ps.DataFrame(valuesM2, columns=bands_name2)

In [49]:
sdftM.shape, sdftM2.shape

((9, 4), (9, 4))

In [12]:
# Create a list of index positions [i, j]
pos_0=np.repeat(np.arange(matrix.shape[0]), matrix.shape[1])
pos_1=np.tile(np.arange(matrix.shape[1]), matrix.shape[0])
#pyspark pandas dataframe doesn't support ndarray
pos_0 = pos_0.tolist()
pos_1 = pos_1.tolist()
type(pos_0)

list

In [50]:
sdftM.insert(0,'coords_0', pos_0)
sdftM.insert(1,'coords_1', pos_1)

#para a segunda matrix
sdftM2.insert(0,'coords_0', pos_0)
sdftM2.insert(1,'coords_1', pos_1)

24/08/14 17:16:04 WARN AttachDistributedSequenceExec: clean up cached RDD(789) in AttachDistributedSequenceExec(14567)
24/08/14 17:16:04 WARN AttachDistributedSequenceExec: clean up cached RDD(820) in AttachDistributedSequenceExec(14827)


In [51]:
sdftM

24/08/14 17:16:08 WARN AttachDistributedSequenceExec: clean up cached RDD(841) in AttachDistributedSequenceExec(15093)


,coords_0,coords_1,b1t1,b2t1,b3t1,b4t1
0,0,0,1.0,2.0,3.0,4.0
1,0,1,-9999.0,6.0,-7.0,8.0
2,0,2,9.0,10.0,11.0,-9999.0
3,1,0,13.0,14.0,15.0,16.0
4,1,1,17.0,18.0,19.0,20.0
5,1,2,21.0,22.0,23.0,24.0
6,2,0,25.0,26.0,27.0,28.0
7,2,1,29.0,30.0,31.0,32.0
8,2,2,33.0,-32768.0,35.0,36.0


In [52]:
sdftM2

24/08/14 17:16:11 WARN AttachDistributedSequenceExec: clean up cached RDD(861) in AttachDistributedSequenceExec(15458)


,coords_0,coords_1,b1t2,b2t2,b3t2,b4t2
0,0,0,-9999.0,22.0,23.0,24.0
1,0,1,25.0,26.0,-27.0,28.0
2,0,2,29.0,40.0,41.0,-9999.0
3,1,0,43.0,44.0,45.0,46.0
4,1,1,47.0,48.0,49.0,50.0
5,1,2,51.0,52.0,53.0,54.0
6,2,0,55.0,56.0,57.0,58.0
7,2,1,59.0,70.0,71.0,72.0
8,2,2,73.0,-32768.0,75.0,76.0


In [68]:
type(sdftM.index)

pyspark.pandas.indexes.numeric.Int64Index

In [53]:
sdftM.insert(0,'orig_index',sdftM.index)
df_toPCA= sdftM#.copy()

In [54]:
df_toPCA

24/08/14 17:16:31 WARN AttachDistributedSequenceExec: clean up cached RDD(881) in AttachDistributedSequenceExec(15874)


,orig_index,coords_0,coords_1,b1t1,b2t1,b3t1,b4t1
0,0,0,0,1.0,2.0,3.0,4.0
1,1,0,1,-9999.0,6.0,-7.0,8.0
2,2,0,2,9.0,10.0,11.0,-9999.0
3,3,1,0,13.0,14.0,15.0,16.0
4,4,1,1,17.0,18.0,19.0,20.0
5,5,1,2,21.0,22.0,23.0,24.0
6,6,2,0,25.0,26.0,27.0,28.0
7,7,2,1,29.0,30.0,31.0,32.0
8,8,2,2,33.0,-32768.0,35.0,36.0


In [55]:
df_toPCA = ps.merge(df_toPCA, sdftM2, on=['coords_0','coords_1'])

In [56]:
df_toPCA

24/08/14 17:16:42 WARN AttachDistributedSequenceExec: clean up cached RDD(901) in AttachDistributedSequenceExec(16359)
24/08/14 17:16:43 WARN AttachDistributedSequenceExec: clean up cached RDD(911) in AttachDistributedSequenceExec(16446)


,orig_index,coords_0,coords_1,b1t1,b2t1,b3t1,b4t1,b1t2,b2t2,b3t2,b4t2
0,0,0,0,1.0,2.0,3.0,4.0,-9999.0,22.0,23.0,24.0
1,1,0,1,-9999.0,6.0,-7.0,8.0,25.0,26.0,-27.0,28.0
2,2,0,2,9.0,10.0,11.0,-9999.0,29.0,40.0,41.0,-9999.0
3,3,1,0,13.0,14.0,15.0,16.0,43.0,44.0,45.0,46.0
4,4,1,1,17.0,18.0,19.0,20.0,47.0,48.0,49.0,50.0
5,5,1,2,21.0,22.0,23.0,24.0,51.0,52.0,53.0,54.0
6,6,2,0,25.0,26.0,27.0,28.0,55.0,56.0,57.0,58.0
7,7,2,1,29.0,30.0,31.0,32.0,59.0,70.0,71.0,72.0
8,8,2,2,33.0,-32768.0,35.0,36.0,73.0,-32768.0,75.0,76.0


In [69]:
type(df_toPCA)

pyspark.pandas.frame.DataFrame

In [70]:
os.getcwd()

'/Users/flaviaschneider/Documents/flavia/Data_GEOBIA/notebooks'

In [75]:
# Salvar spark df_toPCA
temp_path= '/Users/flaviaschneider/Documents/flavia/Data_GEOBIA' + '/data/tmp'
sdfPath = temp_path + "/sdf_toPCA"
df_toPCA.to_parquet(
    sdfPath,
    mode = 'overwrite',
    )

24/08/14 18:32:06 WARN AttachDistributedSequenceExec: clean up cached RDD(1541) in AttachDistributedSequenceExec(31338)
24/08/14 18:32:06 WARN AttachDistributedSequenceExec: clean up cached RDD(1551) in AttachDistributedSequenceExec(31425)


In [77]:
df_toPCA_read = ps.read_parquet(sdfPath)
df_toPCA_read

/Users/flaviaschneider/Documents/flavia/Data_GEOBIA/.venv_data_geobia/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_parquet`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


,orig_index,coords_0,coords_1,b1t1,b2t1,b3t1,b4t1,b1t2,b2t2,b3t2,b4t2
0,3,1,0,13.0,14.0,15.0,16.0,43.0,44.0,45.0,46.0
1,4,1,1,17.0,18.0,19.0,20.0,47.0,48.0,49.0,50.0
2,5,1,2,21.0,22.0,23.0,24.0,51.0,52.0,53.0,54.0
3,6,2,0,25.0,26.0,27.0,28.0,55.0,56.0,57.0,58.0
4,7,2,1,29.0,30.0,31.0,32.0,59.0,70.0,71.0,72.0


## Gen spark df to PCA

In [57]:
# Convert to PySpark DataFrame
spark_df = df_toPCA.to_spark()

/Users/flaviaschneider/Documents/flavia/Data_GEOBIA/.venv_data_geobia/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [76]:
#saving Pyspar Dataframe to parquet
temp_path= '/Users/flaviaschneider/Documents/flavia/Data_GEOBIA' + '/data/tmp'
sdfPath = temp_path + "/spark_df_toPCA"
spark_df.write.parquet(sdfPath)

24/08/14 18:34:39 WARN AttachDistributedSequenceExec: clean up cached RDD(1590) in AttachDistributedSequenceExec(32630)
24/08/14 18:34:40 WARN AttachDistributedSequenceExec: clean up cached RDD(1600) in AttachDistributedSequenceExec(32717)


In [79]:
spark_df_read= spark.read.parquet(sdfPath)

In [80]:
spark_df_read.show()

+----------+--------+--------+----+----+----+----+----+----+----+----+
|orig_index|coords_0|coords_1|b1t1|b2t1|b3t1|b4t1|b1t2|b2t2|b3t2|b4t2|
+----------+--------+--------+----+----+----+----+----+----+----+----+
|         3|       1|       0|13.0|14.0|15.0|16.0|43.0|44.0|45.0|46.0|
|         4|       1|       1|17.0|18.0|19.0|20.0|47.0|48.0|49.0|50.0|
|         5|       1|       2|21.0|22.0|23.0|24.0|51.0|52.0|53.0|54.0|
|         6|       2|       0|25.0|26.0|27.0|28.0|55.0|56.0|57.0|58.0|
|         7|       2|       1|29.0|30.0|31.0|32.0|59.0|70.0|71.0|72.0|
+----------+--------+--------+----+----+----+----+----+----+----+----+



In [72]:
spark_df.show()

24/08/14 18:29:40 WARN AttachDistributedSequenceExec: clean up cached RDD(1401) in AttachDistributedSequenceExec(27815)
24/08/14 18:29:40 WARN AttachDistributedSequenceExec: clean up cached RDD(1411) in AttachDistributedSequenceExec(27902)


+----------+--------+--------+----+----+----+----+----+----+----+----+
|orig_index|coords_0|coords_1|b1t1|b2t1|b3t1|b4t1|b1t2|b2t2|b3t2|b4t2|
+----------+--------+--------+----+----+----+----+----+----+----+----+
|         3|       1|       0|13.0|14.0|15.0|16.0|43.0|44.0|45.0|46.0|
|         4|       1|       1|17.0|18.0|19.0|20.0|47.0|48.0|49.0|50.0|
|         5|       1|       2|21.0|22.0|23.0|24.0|51.0|52.0|53.0|54.0|
|         6|       2|       0|25.0|26.0|27.0|28.0|55.0|56.0|57.0|58.0|
|         7|       2|       1|29.0|30.0|31.0|32.0|59.0|70.0|71.0|72.0|
+----------+--------+--------+----+----+----+----+----+----+----+----+



In [58]:
spark_df.show()

24/08/14 17:16:58 WARN AttachDistributedSequenceExec: clean up cached RDD(953) in AttachDistributedSequenceExec(17526)
24/08/14 17:16:58 WARN AttachDistributedSequenceExec: clean up cached RDD(963) in AttachDistributedSequenceExec(17613)


+----------+--------+--------+-------+--------+----+-------+-------+--------+-----+-------+
|orig_index|coords_0|coords_1|   b1t1|    b2t1|b3t1|   b4t1|   b1t2|    b2t2| b3t2|   b4t2|
+----------+--------+--------+-------+--------+----+-------+-------+--------+-----+-------+
|         0|       0|       0|    1.0|     2.0| 3.0|    4.0|-9999.0|    22.0| 23.0|   24.0|
|         1|       0|       1|-9999.0|     6.0|-7.0|    8.0|   25.0|    26.0|-27.0|   28.0|
|         2|       0|       2|    9.0|    10.0|11.0|-9999.0|   29.0|    40.0| 41.0|-9999.0|
|         3|       1|       0|   13.0|    14.0|15.0|   16.0|   43.0|    44.0| 45.0|   46.0|
|         4|       1|       1|   17.0|    18.0|19.0|   20.0|   47.0|    48.0| 49.0|   50.0|
|         5|       1|       2|   21.0|    22.0|23.0|   24.0|   51.0|    52.0| 53.0|   54.0|
|         6|       2|       0|   25.0|    26.0|27.0|   28.0|   55.0|    56.0| 57.0|   58.0|
|         7|       2|       1|   29.0|    30.0|31.0|   32.0|   59.0|    70.0| 71

In [59]:
spark_df_coords = spark_df
#%% #replace nans
t1=time.time()
# Replace -9999 with NaN in all columns
spark_df = spark_df.select([when(col(c) == -9999, F.lit(None)).otherwise(col(c)).alias(c) for c in spark_df.columns])
spark_df = spark_df.select([when(col(c) == -32768, F.lit(None)).otherwise(col(c)).alias(c) for c in spark_df.columns])
t2=time.time()
t2-t1

0.19775795936584473

In [60]:
# #drop the nans
spark_df = spark_df.na.drop()
spark_df.show()

24/08/14 17:17:10 WARN AttachDistributedSequenceExec: clean up cached RDD(1002) in AttachDistributedSequenceExec(18614)
24/08/14 17:17:10 WARN AttachDistributedSequenceExec: clean up cached RDD(1012) in AttachDistributedSequenceExec(18701)


+----------+--------+--------+----+----+----+----+----+----+----+----+
|orig_index|coords_0|coords_1|b1t1|b2t1|b3t1|b4t1|b1t2|b2t2|b3t2|b4t2|
+----------+--------+--------+----+----+----+----+----+----+----+----+
|         3|       1|       0|13.0|14.0|15.0|16.0|43.0|44.0|45.0|46.0|
|         4|       1|       1|17.0|18.0|19.0|20.0|47.0|48.0|49.0|50.0|
|         5|       1|       2|21.0|22.0|23.0|24.0|51.0|52.0|53.0|54.0|
|         6|       2|       0|25.0|26.0|27.0|28.0|55.0|56.0|57.0|58.0|
|         7|       2|       1|29.0|30.0|31.0|32.0|59.0|70.0|71.0|72.0|
+----------+--------+--------+----+----+----+----+----+----+----+----+



In [61]:
spark_df_coords.show()

24/08/14 17:17:30 WARN AttachDistributedSequenceExec: clean up cached RDD(1051) in AttachDistributedSequenceExec(19738)
24/08/14 17:17:30 WARN AttachDistributedSequenceExec: clean up cached RDD(1061) in AttachDistributedSequenceExec(19825)


+----------+--------+--------+-------+--------+----+-------+-------+--------+-----+-------+
|orig_index|coords_0|coords_1|   b1t1|    b2t1|b3t1|   b4t1|   b1t2|    b2t2| b3t2|   b4t2|
+----------+--------+--------+-------+--------+----+-------+-------+--------+-----+-------+
|         0|       0|       0|    1.0|     2.0| 3.0|    4.0|-9999.0|    22.0| 23.0|   24.0|
|         1|       0|       1|-9999.0|     6.0|-7.0|    8.0|   25.0|    26.0|-27.0|   28.0|
|         2|       0|       2|    9.0|    10.0|11.0|-9999.0|   29.0|    40.0| 41.0|-9999.0|
|         3|       1|       0|   13.0|    14.0|15.0|   16.0|   43.0|    44.0| 45.0|   46.0|
|         4|       1|       1|   17.0|    18.0|19.0|   20.0|   47.0|    48.0| 49.0|   50.0|
|         5|       1|       2|   21.0|    22.0|23.0|   24.0|   51.0|    52.0| 53.0|   54.0|
|         6|       2|       0|   25.0|    26.0|27.0|   28.0|   55.0|    56.0| 57.0|   58.0|
|         7|       2|       1|   29.0|    30.0|31.0|   32.0|   59.0|    70.0| 71

In [62]:
bands_name+bands_name2

['b1t1', 'b2t1', 'b3t1', 'b4t1', 'b1t2', 'b2t2', 'b3t2', 'b4t2']

In [85]:
spark_df_coords.columns[3:]

['b1t1', 'b2t1', 'b3t1', 'b4t1', 'b1t2', 'b2t2', 'b3t2', 'b4t2']

In [63]:
# assembler nao suporta nan's
assembler = VectorAssembler(inputCols=bands_name+bands_name2, outputCol="features")
df_with_features=assembler.transform(spark_df)
print (df_with_features.show())
df_with_features.dtypes

24/08/14 17:17:40 WARN AttachDistributedSequenceExec: clean up cached RDD(1100) in AttachDistributedSequenceExec(20807)
24/08/14 17:17:41 WARN AttachDistributedSequenceExec: clean up cached RDD(1110) in AttachDistributedSequenceExec(20894)


+----------+--------+--------+----+----+----+----+----+----+----+----+--------------------+
|orig_index|coords_0|coords_1|b1t1|b2t1|b3t1|b4t1|b1t2|b2t2|b3t2|b4t2|            features|
+----------+--------+--------+----+----+----+----+----+----+----+----+--------------------+
|         3|       1|       0|13.0|14.0|15.0|16.0|43.0|44.0|45.0|46.0|[13.0,14.0,15.0,1...|
|         4|       1|       1|17.0|18.0|19.0|20.0|47.0|48.0|49.0|50.0|[17.0,18.0,19.0,2...|
|         5|       1|       2|21.0|22.0|23.0|24.0|51.0|52.0|53.0|54.0|[21.0,22.0,23.0,2...|
|         6|       2|       0|25.0|26.0|27.0|28.0|55.0|56.0|57.0|58.0|[25.0,26.0,27.0,2...|
|         7|       2|       1|29.0|30.0|31.0|32.0|59.0|70.0|71.0|72.0|[29.0,30.0,31.0,3...|
+----------+--------+--------+----+----+----+----+----+----+----+----+--------------------+

None


[('orig_index', 'bigint'),
 ('coords_0', 'bigint'),
 ('coords_1', 'bigint'),
 ('b1t1', 'double'),
 ('b2t1', 'double'),
 ('b3t1', 'double'),
 ('b4t1', 'double'),
 ('b1t2', 'double'),
 ('b2t2', 'double'),
 ('b3t2', 'double'),
 ('b4t2', 'double'),
 ('features', 'vector')]

## Run PCA

In [64]:
# Apply PCA
pca = PCA(k=4, inputCol="features", outputCol="pca_features")
pca_model = pca.fit(df_with_features)
df_with_pca = pca_model.transform(df_with_features)
df_with_pca.show()

24/08/14 17:17:57 WARN AttachDistributedSequenceExec: clean up cached RDD(1149) in AttachDistributedSequenceExec(22019)
24/08/14 17:17:57 WARN AttachDistributedSequenceExec: clean up cached RDD(1159) in AttachDistributedSequenceExec(22106)
24/08/14 17:17:59 WARN AttachDistributedSequenceExec: clean up cached RDD(1205) in AttachDistributedSequenceExec(23212)
24/08/14 17:17:59 WARN AttachDistributedSequenceExec: clean up cached RDD(1215) in AttachDistributedSequenceExec(23299)


+----------+--------+--------+----+----+----+----+----+----+----+----+--------------------+--------------------+
|orig_index|coords_0|coords_1|b1t1|b2t1|b3t1|b4t1|b1t2|b2t2|b3t2|b4t2|            features|        pca_features|
+----------+--------+--------+----+----+----+----+----+----+----+----+--------------------+--------------------+
|         3|       1|       0|13.0|14.0|15.0|16.0|43.0|44.0|45.0|46.0|[13.0,14.0,15.0,1...|[-89.015368600145...|
|         4|       1|       1|17.0|18.0|19.0|20.0|47.0|48.0|49.0|50.0|[17.0,18.0,19.0,2...|[-100.02516560190...|
|         5|       1|       2|21.0|22.0|23.0|24.0|51.0|52.0|53.0|54.0|[21.0,22.0,23.0,2...|[-111.03496260366...|
|         6|       2|       0|25.0|26.0|27.0|28.0|55.0|56.0|57.0|58.0|[25.0,26.0,27.0,2...|[-122.04475960542...|
|         7|       2|       1|29.0|30.0|31.0|32.0|59.0|70.0|71.0|72.0|[29.0,30.0,31.0,3...|[-146.52870730930...|
+----------+--------+--------+----+----+----+----+----+----+----+----+--------------------+-----

In [65]:
# Show the PCA results
df_with_pca.select("pca_features").show(truncate=False)

24/08/14 17:18:00 WARN AttachDistributedSequenceExec: clean up cached RDD(1254) in AttachDistributedSequenceExec(24310)
24/08/14 17:18:01 WARN AttachDistributedSequenceExec: clean up cached RDD(1264) in AttachDistributedSequenceExec(24397)


+-------------------------------------------------------------------------------+
|pca_features                                                                   |
+-------------------------------------------------------------------------------+
|[-89.01536860014505,13.837057237010828,-7.895133277410458,-9.37017629851543]   |
|[-100.02516560190612,11.232374918567228,-7.895133277410445,-9.370176298515425] |
|[-111.03496260366722,8.627692600123641,-7.89513327741043,-9.370176298515428]   |
|[-122.04475960542831,6.0230102816800475,-7.8951332774104195,-9.370176298515426]|
|[-146.52870730930695,14.301675924990885,-7.895133277410457,-9.370176298515426] |
+-------------------------------------------------------------------------------+



In [66]:
pca_model.explainedVariance

DenseVector([0.9754, 0.0246, 0.0, 0.0])

In [67]:
pca_model.transform(df_with_features).collect()[1].pca_features

24/08/14 17:18:10 WARN AttachDistributedSequenceExec: clean up cached RDD(1303) in AttachDistributedSequenceExec(25486)
24/08/14 17:18:10 WARN AttachDistributedSequenceExec: clean up cached RDD(1313) in AttachDistributedSequenceExec(25573)


DenseVector([-100.0252, 11.2324, -7.8951, -9.3702])

In [86]:
pca_model

PCAModel: uid=PCA_756454922838, k=4

In [87]:
#saving Pyspar Dataframe to parquet
tmp_path= '/Users/flaviaschneider/Documents/flavia/Data_GEOBIA' + '/data/tmp'
sdfPath = tmp_path + "/df_with_pca"
df_with_pca.write.parquet(sdfPath)

24/08/16 11:53:20 WARN AttachDistributedSequenceExec: clean up cached RDD(1654) in AttachDistributedSequenceExec(33803)
24/08/16 11:53:20 WARN AttachDistributedSequenceExec: clean up cached RDD(1664) in AttachDistributedSequenceExec(33890)


In [94]:
type(df_with_pca), df_with_pca.show()

24/08/16 11:55:55 WARN AttachDistributedSequenceExec: clean up cached RDD(1707) in AttachDistributedSequenceExec(35030)
24/08/16 11:55:55 WARN AttachDistributedSequenceExec: clean up cached RDD(1717) in AttachDistributedSequenceExec(35117)


+----------+--------+--------+----+----+----+----+----+----+----+----+--------------------+--------------------+
|orig_index|coords_0|coords_1|b1t1|b2t1|b3t1|b4t1|b1t2|b2t2|b3t2|b4t2|            features|        pca_features|
+----------+--------+--------+----+----+----+----+----+----+----+----+--------------------+--------------------+
|         3|       1|       0|13.0|14.0|15.0|16.0|43.0|44.0|45.0|46.0|[13.0,14.0,15.0,1...|[-89.015368600145...|
|         4|       1|       1|17.0|18.0|19.0|20.0|47.0|48.0|49.0|50.0|[17.0,18.0,19.0,2...|[-100.02516560190...|
|         5|       1|       2|21.0|22.0|23.0|24.0|51.0|52.0|53.0|54.0|[21.0,22.0,23.0,2...|[-111.03496260366...|
|         6|       2|       0|25.0|26.0|27.0|28.0|55.0|56.0|57.0|58.0|[25.0,26.0,27.0,2...|[-122.04475960542...|
|         7|       2|       1|29.0|30.0|31.0|32.0|59.0|70.0|71.0|72.0|[29.0,30.0,31.0,3...|[-146.52870730930...|
+----------+--------+--------+----+----+----+----+----+----+----+----+--------------------+-----

(pyspark.sql.dataframe.DataFrame, None)

## Segmentacao

In [95]:
df_with_pca_read = spark.read.parquet(sdfPath)

In [98]:
df_with_pca_read.select('pca_features').show()

+--------------------+
|        pca_features|
+--------------------+
|[-89.015368600145...|
|[-100.02516560190...|
|[-111.03496260366...|
|[-122.04475960542...|
|[-146.52870730930...|
+--------------------+



In [100]:
df_with_pca_read.show(n=3)

+----------+--------+--------+----+----+----+----+----+----+----+----+--------------------+--------------------+
|orig_index|coords_0|coords_1|b1t1|b2t1|b3t1|b4t1|b1t2|b2t2|b3t2|b4t2|            features|        pca_features|
+----------+--------+--------+----+----+----+----+----+----+----+----+--------------------+--------------------+
|         3|       1|       0|13.0|14.0|15.0|16.0|43.0|44.0|45.0|46.0|[13.0,14.0,15.0,1...|[-89.015368600145...|
|         4|       1|       1|17.0|18.0|19.0|20.0|47.0|48.0|49.0|50.0|[17.0,18.0,19.0,2...|[-100.02516560190...|
|         5|       1|       2|21.0|22.0|23.0|24.0|51.0|52.0|53.0|54.0|[21.0,22.0,23.0,2...|[-111.03496260366...|
+----------+--------+--------+----+----+----+----+----+----+----+----+--------------------+--------------------+
only showing top 3 rows



In [45]:
#spark.stop()